# Setup

In [11]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

In [12]:
import numpy as np

from src.data import load_train_data
from src.features import add_engineered_features

data = load_train_data()
X = add_engineered_features(data)
y = np.log1p(data['SalePrice'])

# Pipeline 

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

nominal_cols = ['MSZoning','MSSubClass', 'Street',  'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType','Foundation', 'Heating','CentralAir', 'Electrical','Functional',
        'GarageFinish','PavedDrive',
       'SaleType', 'SaleCondition'
]

ordinal_cols = ['OverallQual','OverallCond','ExterQual', 'ExterCond',
                'HeatingQC','KitchenQual'

]
num_structured_missing = ['GarageYrBlt']
num_unstruc_missing = ['LotFrontage']

rest_num_cols = [
  col for col in X.columns
  if col not in nominal_cols + ordinal_cols + ["SalePrice"] + num_structured_missing + num_unstruc_missing
]

# pipelines

# for numerical attributes with structured missingness, value 0 will be imputed 
num_struc_missing_pipeline = Pipeline([
  ('imputer',
   SimpleImputer(strategy='constant',
   fill_value = 0)),
  ('scaler',StandardScaler())
])

# for numerical attributes with true missingness, and the rest of the num cols, the median value will be imputed 
rest_num = Pipeline([
  ('imputer',
   SimpleImputer(strategy='median')),
   ('scaler',StandardScaler())
])

# Only few columns need encoding now 
Qual_and_Con_scale = ['None','Po','Fa','TA','Gd','Ex']
qnc_cols = ['ExterQual','ExterCond','HeatingQC','KitchenQual']

# Generate mappings
qnc_map = {k: i for i, k in enumerate(Qual_and_Con_scale)}

# apply the mappings
for col in qnc_cols:
  X[col] = X[col].map(qnc_map)

# impute ordinal (dummy)
ordinal_pipeline = Pipeline([
  ('imputer', SimpleImputer(strategy='most_frequent'))
])

# preprocessor
preprocessor = ColumnTransformer([
  ('num_struc',num_struc_missing_pipeline,
    num_structured_missing),
  ('rest_num', rest_num, 
   rest_num_cols),
  ('nominal', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
  ('ordinal',ordinal_pipeline,ordinal_cols)
])


# Evaluation Function

In [14]:
def evaluate_model(model, X, y, cv=5):
  pipeline = Pipeline([
    ('prep', preprocessor),
    ('model',model)
  ])

  scores = cross_val_score(
    pipeline,
    X,
    y,
    scoring = 'neg_root_mean_squared_error',
    cv =cv
  )

  return {
    'rmse': -scores,
    'rmse_mean': -scores.mean(),
    'rmse_std': scores.std()
  }

# Model comparison

I will first compare ridge, lasso, and elasticnet, three ways of regularisation

Each type of regularisation behaves differently, for Ridge, it shrinks coefficients smoothly, so it has low sensitively of alpha. On the other extreme, Lasso can force coefficients to become 0, and it is very sensitive to alpha. While ElasticNet is in between 

I will start off with the common values for default models

Other than linear models, I will also try random forest

In [15]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor

models= {
  'Ridge' : Ridge(alpha=1.0),
  'Lasso' : Lasso(alpha=0.001),
  'ElasticNet' : ElasticNet(alpha=0.01, l1_ratio=0.5),

  'rfr': RandomForestRegressor(
  n_estimators=300,
  max_depth=None, # it might overfit
  random_state=42,
  n_jobs=-1
)
}


In [ ]:
results = {}

for name, model in models.items():
    result = evaluate_model(model, X, y)
    results[name] = result
    print(f"{name}: RMSE = {result['rmse_mean']:.f} ± {result['rmse_std']:.4f}")

Ridge: RMSE = 0.1411 ± 0.0224
Lasso: RMSE = 0.1371 ± 0.0271
ElasticNet: RMSE = 0.1501 ± 0.0254
rfr: RMSE = 0.1429 ± 0.0083


- Ridge - Good mean RMSE, good uncertainty 
- Lasso - Best mean RMSE, worst uncertainty, it is sensitive to data splits and it is the most instable, though have the best RMSE
- ElasticNet - Worst RMSE, moderate uncertainty, maybe not so useful at this stage 
- RandomForestRegressor - Moderate RMSE, best uncertainty - Very stable, and RMSE is not bad